# TRACER — Colab Bootstrap (Module 1)

Run this once at the start of every Colab session. It:
1. Mounts Google Drive (persistent storage across sessions)
2. Clones (or pulls) the TRACER GitHub repo
3. Sets up the Drive folder convention for datasets/models
4. Installs the ai-engine dependencies
5. Verifies GPU, library versions, and does one dummy CLIP forward pass

**Before running:** replace `<your-username>` below with your actual GitHub username, and make
sure the TRACER repo is already created on GitHub (see Module 1 docs for `git init` steps).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/TRACER"
for sub in ["datasets", "models", "outputs"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

print("Drive folders ready:")
for sub in ["datasets", "models", "outputs"]:
    print(f" - {DRIVE_ROOT}/{sub}")


In [ ]:
GITHUB_USERNAME = "<your-username>"   # <-- change this
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/TRACER.git"

import os
if not os.path.exists("/content/TRACER"):
    !git clone {REPO_URL} /content/TRACER
else:
    %cd /content/TRACER
    !git pull

%cd /content/TRACER/ai-engine


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
# --- Environment verification ---
import torch, transformers, diffusers, sklearn

print("=" * 50)
print("TRACER AI ENGINE — ENVIRONMENT CHECK")
print("=" * 50)
print(f"Torch version:        {torch.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")
print(f"Transformers version: {transformers.__version__}")
print(f"Diffusers version:    {diffusers.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")


In [ ]:
# --- Dummy CLIP forward pass to confirm the VLM loads correctly ---
from transformers import CLIPModel, CLIPProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

dummy_image = torch.rand(1, 3, 224, 224)
inputs = processor(images=dummy_image, return_tensors="pt", do_rescale=False)
with torch.no_grad():
    output = model.get_image_features(pixel_values=inputs["pixel_values"].to(device))

# transformers versions differ on whether get_image_features returns a plain tensor
# or a ModelOutput object — handle both so this cell is future-proof.
if isinstance(output, torch.Tensor):
    features = output
elif hasattr(output, "pooler_output"):
    features = output.pooler_output
elif hasattr(output, "image_embeds"):
    features = output.image_embeds
else:
    raise TypeError(f"Unexpected output type from get_image_features: {type(output)}")

print(f"CLIP loaded successfully on {device}. Feature shape: {features.shape}")
print("Module 1 environment check PASSED.")


## Next Step

If everything above ran without errors, your environment is ready. Continue to
**Module 2 — Vision-Language Model Integration**.